In [1]:
#SCALAR GAUSS BONNET MODIFIED EXAMPLE

# Isotropic Schwarzschild BH example
# see further details in https://github.com/GRChombo/engrenage/wiki/Running-the-black-hole-example

# restart the kernel to clear past work
# (can also do this manually from the Kernel options above)
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [2]:
# load the required python modules
import numpy as np
from scipy.interpolate import interp1d
from scipy.integrate import odeint
from scipy.integrate import solve_ivp
import time
import sys
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

# homemade source code from source folder
sys.path.append('/Users/serdaryildiz/Desktop/GRSerdar/engrenage/SGB_BR/source')

from initialdata.bhinitialconditions import *
from backgrounds.sphericalbackground import *
from bssn.constraintsdiagnostic import *
from bssn.ahfinder import *
from core.rhsevolution_MG_testing import *
from core.grid import Grid
from core.spacing import *
from core.display import *
from core.statevector import *
from matter.scalarmatter_MG import *

In [3]:
# Set up the chosen matter class
#scalar_mu = 1.0
scalar_mu = 0 # in the EsGB case
my_matter = ScalarMatter(scalar_mu)
my_state_vector = StateVector(my_matter)

# Input parameters for grid and evolution here
r_max = 96.0 # outer edge of the grid (including ghosts)
#min_dr = 1 / 16 # roughly 32 points across the BH

min_dr = 1/16
#min_dr = 1 / 64
max_dr = 2

# SinhSpacing
# params = SinhSpacing.get_parameters(r_max, min_dr, max_dr)
# spacing = SinhSpacing(**params)

# CubicSpacing
params = CubicSpacing.get_parameters(r_max, min_dr, max_dr)
spacing = CubicSpacing(**params)

grid = Grid(spacing, my_state_vector)
r = grid.r
num_points = r.size
background = FlatSphericalBackground(r)

# Check grid params
print(params)

{'r_max': 96.0, 'extent': <SpacingExtent.HALF: 0>, 'a': 1.5213625583444466, 'num_points': 140}


In [4]:
initial_state = get_initial_state(grid, background)

#unpackage the vector for readability
(initial_phi, initial_hrr, initial_htt, initial_hpp, 
 initial_K, initial_arr, initial_att, initial_app, 
 initial_lambdar, initial_shiftr, initial_br, initial_lapse, 
 initial_u, initial_v) = np.array_split(initial_state, grid.NUM_VARS)

In [5]:
print(len(initial_state))

1960


In [6]:
"""
# check the Hamiltonian and momentum constraints are initially satisfied
Ham, Mom = get_constraints_diagnostic(initial_state, np.array([0]), grid, background, my_matter)

# plot the profile for Ham
#plt.plot(r, Ham[0])
plt.plot(r, Mom[0])

plt.xlabel('r')
plt.ylabel('Ham value')
ax = plt.gcf().gca()
set_grid_on_ax(ax, r)
plt.xlim(-0.5,10)
plt.ylim(-0.1,0.1)
"""

"\n# check the Hamiltonian and momentum constraints are initially satisfied\nHam, Mom = get_constraints_diagnostic(initial_state, np.array([0]), grid, background, my_matter)\n\n# plot the profile for Ham\n#plt.plot(r, Ham[0])\nplt.plot(r, Mom[0])\n\nplt.xlabel('r')\nplt.ylabel('Ham value')\nax = plt.gcf().gca()\nset_grid_on_ax(ax, r)\nplt.xlim(-0.5,10)\nplt.ylim(-0.1,0.1)\n"

In [7]:
# Check that for a higher resolution the Ham constraint converges (the Mom constraint is trivially zero)

# Input parameters for HR grid
min_dr_HR = 1 / 32 # roughly 64 points across the BH
max_dr_HR = 1

# CubicSpacing
params_HR = CubicSpacing.get_parameters(r_max, min_dr_HR, max_dr_HR)
spacing_HR = CubicSpacing(**params_HR)

grid_HR = Grid(spacing_HR, my_state_vector)
r_HR = grid_HR.r
num_points_HR = r_HR.size
background_HR = FlatSphericalBackground(r_HR)

# Check grid params
print(params_HR)

# Get initial state
initial_state_HR = get_initial_state(grid_HR, background_HR)



{'r_max': 96.0, 'extent': <SpacingExtent.HALF: 0>, 'a': 1.5213625583444466, 'num_points': 275}


In [8]:
# find the apparent horizon for a given set of states
omega, ah_radius, bh_mass = get_horizon_diagnostics(initial_state, np.array([0]), grid, background, my_matter)

print("horizon is at r =", round(ah_radius[0],1))
print("horizon is at r =", ah_radius[0])

horizon is at r = 0.5
horizon is at r = 0.498678538330956


# Start running the simulation

In [9]:
# for control of time integrator and spatial grid
#T = 10 # Maximum evolution time
T = 1
#num_points_t = 128 # time resolution (only for outputs, not for integration, which is decided by python)
num_points_t = 250

# Work out dt and time spacing of outputs
dt = T/num_points_t
t = np.linspace(0, T-dt, num_points_t)

# Solve for the solution using RK45 integration of the ODE
# to make like (older) python odeint method use method='LSODA' instead
# use tqdm package to track progress
with tqdm(total=1000, unit="‰") as progress_bar:
    dense_solution = solve_ivp(get_rhs, [0,T], initial_state, 
                               args=(grid, background, my_matter, progress_bar, [0, T/1000],0.2, 0.4, 0.05),
                        #atol=1e-5, rtol=1e-5,
                        max_step = 0.4 * min_dr, #for stability and for KO coeff of 1
                        method='RK45', dense_output=True)

# Interpolate the solution at the time points requested
solution = dense_solution.sol(t).T

  0%|          | 0/1000 [00:00<?, ?‰/s]

In [10]:
inta = 0
for i in range(len(my_state_vector.VARIABLE_NAMES)):
    print('Index: ', inta, 'Variable Name: ',  my_state_vector.VARIABLE_NAMES[i])
    inta = inta +1

Index:  0 Variable Name:  phi
Index:  1 Variable Name:  hrr
Index:  2 Variable Name:  htt
Index:  3 Variable Name:  hpp
Index:  4 Variable Name:  K
Index:  5 Variable Name:  arr
Index:  6 Variable Name:  att
Index:  7 Variable Name:  app
Index:  8 Variable Name:  lambdar
Index:  9 Variable Name:  shiftr
Index:  10 Variable Name:  br
Index:  11 Variable Name:  lapse
Index:  12 Variable Name:  u
Index:  13 Variable Name:  v


In [11]:
hat_gamma = np.zeros((3,3))
hat_gamma[1,1] = r[10]**2
hat_gamma[2,2] = r[10]**2

print(hat_gamma)

[[0.        0.        0.       ]
 [0.        0.2303386 0.       ]
 [0.        0.        0.2303386]]


# Initial Values

In [12]:
#print(initial[200][0][0][10])
#print(initial[i][j][0][10])

i,j,k = 200, 0, 10

#vars = ["chi", "hrr", "htt", "hpp", "K", "arr", "att", "app", "lambdar", "shiftr", "lapse","phi", "Pi", "dchi", "dphi", "dPi", "dlapse", "dK", "dgamma", "dA", "dLambda", "dshift", "d2chi", "d2phi", "d2lapse", "d2gamma", "d2shift"]
vars = ["chi", "Gamma", "K", "A", "Lambdar", "Shiftr", "lapse","phi", "Pi", "dchi", "dphi", "dPi", "dlapse", "dK", "dGamma", "dA", "dLambda", "dShift", "d2chi", "d2phi", "d2lapse", "d2Gamma", "d2Shift"]
time_idx = i
time = t[i]

var_idx = j
var = vars[j]

pos_idx = k
pos = r[pos_idx]


print( "time: ", time,"position: ", pos)
print('')
for m in range(len(vars)):
    print(vars[m],"= ",initial[i][m][0][k],";" )


time:  0.8 position:  0.47993604100070275

chi =  0.0609867684537276 ;
Gamma =  [[1.06032395876898, 0.0, 0.0], [0.0, 0.22369044939567875, 0.0], [0.0, 0.0, 0.22369044939567875]] ;
K =  0.01693349180876882 ;
A =  [[-0.22158186368260224, 0.0, 0.0], [0.0, 0.023372338371934, 0.0], [0.0, 0.0, 0.023372338371934]] ;
Lambdar =  [0.4814483211105061, 0.0, 0.0] ;
Shiftr =  [0.09142465934210317, 0.0, 0.0] ;
lapse =  0.9941885069169928 ;
phi =  6.750501947627349e-06 ;
Pi =  0.00013524018932775466 ;
dchi =  [0.24956161159071186, -0.0, -0.0] ;
dphi =  [0.0006706916626822981, 0.0, 0.0] ;
dPi =  [0.008924708167139975, 0.0, 0.0] ;
dlapse =  [-0.0005295522048608529, 0.0, 0.0] ;
dK =  [-0.00014152793581781282, 0.0, 0.0] ;
dGamma =  [[[0.13602221495888256, 0.0, 0.0], [0.0, 0.9178199149090275, 0.0], [0.0, 0.0, 0.9178199149090275]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]] ;
dA =  [[[-0.024257616759648142, 0.0, 0.0], [0.0, 0.09546185007456619, 0

In [13]:

haha = initial[i][9][0][k]
print(haha)

[0.24956161159071186, -0.0, -0.0]


# Coordinate transformation

In [14]:
def point_from_r(r, theta=None, phi=None):
    """
    Return a Cartesian point (x,y,z) on the sphere of radius r.
    - If theta, phi are given (radians), use spherical angles:
        x = r sinθ cosφ, y = r sinθ sinφ, z = r cosθ
    - Otherwise default to x-axis: (r, 0, 0).
    """
    if theta is None or phi is None:
        return float(r), 0.0, 0.0
    s = np.sin(theta)
    return r * s * np.cos(phi), r * s * np.sin(phi), r * np.cos(theta)

# print(point_from_r(r[10]))


def unit_radial(x, y, z, eps=1e-15):
    r = np.sqrt(x*x + y*y + z*z)
    if r < eps:
        raise ValueError("r≈0: direction n is undefined at the origin.")
    n = np.array([x, y, z]) / r
    I = np.eye(3)
    P = I - np.outer(n, n)  # projector to tangential plane
    return r, n, I, P

radius, n, I, P = unit_radial(r[10], 0,0)

In [15]:
for i in range(len(vars)):
    print("index: ", i, vars[i])

index:  0 chi
index:  1 Gamma
index:  2 K
index:  3 A
index:  4 Lambdar
index:  5 Shiftr
index:  6 lapse
index:  7 phi
index:  8 Pi
index:  9 dchi
index:  10 dphi
index:  11 dPi
index:  12 dlapse
index:  13 dK
index:  14 dGamma
index:  15 dA
index:  16 dLambda
index:  17 dShift
index:  18 d2chi
index:  19 d2phi
index:  20 d2lapse
index:  21 d2Gamma
index:  22 d2Shift


# Scalars 

In [16]:
cartesian_names = []
cartesian_objects = [] 

In [17]:
chi = initial[200][0][0][10]
cartesian_names.append("chi")
cartesian_objects.append(chi)

K = initial[200][2][0][10]
cartesian_names.append("K")
cartesian_objects.append(K)

lapse = initial[200][6][0][10]
cartesian_names.append("lapse")
cartesian_objects.append(lapse)

phi = initial[200][7][0][10]
cartesian_names.append("phi")
cartesian_objects.append(phi)

Pi = initial[200][8][0][10]
cartesian_names.append("Pi")
cartesian_objects.append(Pi)

# Derivatives of scalars coordinate transformation

In [18]:
d1chi = initial[200][9][0][10]
cartesian_names.append("dchi")
cartesian_objects.append(d1chi)

d2chi = initial[200][18][0][10]
cartesian_names.append("d2chi")
cartesian_objects.append(d2chi)

d1phi = initial[200][10][0][10]
cartesian_names.append("dphi")
cartesian_objects.append(d1phi)

d2phi = initial[200][19][0][10]
cartesian_names.append("d2phi")
cartesian_objects.append(d2phi)

d1Pi = initial[200][11][0][10]
cartesian_names.append("dPi")
cartesian_objects.append(d1Pi)

d1lapse = initial[200][12][0][10]
cartesian_names.append("dlapse")
cartesian_objects.append(d1lapse)

d2lapse = initial[200][20][0][10]
cartesian_names.append("d2lapse")
cartesian_objects.append(d2lapse)


d1K = initial[200][13][0][10]
cartesian_names.append("dK")
cartesian_objects.append(d1K)


# Vectors coordinate transformations

In [19]:
Lambda = initial[200][4][0][10]
cartesian_names.append("gamma")
cartesian_objects.append(Lambda)

Shift  = initial[200][5][0][10]
cartesian_names.append("shift")
cartesian_objects.append(Shift)


# Derivatives of vectors coordinate transformation

In [20]:
d1Lambda = initial[200][16][0][10]
d1Shift = initial[200][17][0][10]
d2Shift = initial[200][22][0][10]

# ---------- Formulas for a radial vector v^j = V_r(r) n^j ----------
# ∂_i v^j = V_r'(r) n_i n^j + (V_r/r)(δ_i^j - n_i n^j)
# ∂_k∂_i v^j = V_r'' n_k n_i n^j
#            + (V_r'/r)[ n_k(δ_i^j - n_i n^j) + n_i(δ_k^j - n_k n^j) + n^j(δ_ki - n_k n_i)]
#            - 2(V_r/r^2) n^j(δ_ki - n_k n_i)

def grad_vector_full(Vr, Vr_r, r, n):
    nn = np.outer(n, n)                     # [i,j]
    return Vr_r * nn + (Vr / r) * (I - nn)  # [i,j] = ∂_i v^j

def hess_vector_full(Vr, Vr_r, Vr_rr, r, n):
    nn = np.outer(n, n)
    D  = I - nn
    # Build terms with einsum; output shape [k,i,j] = ∂_k ∂_i v^j
    term1 = Vr_rr * np.einsum('k,i,j->kij', n, n, n)
    t2a   = np.einsum('k,ij->kij', n, (I - nn))              # n_k (δ_i^j - n_i n^j)
    t2b   = np.einsum('i,kj->kij', n, (I - nn))              # n_i (δ_k^j - n_k n^j)
    t2c   = np.einsum('j,ki->kij', n, (I - np.outer(n, n)))  # n^j (δ_ki - n_k n_i)
    term2 = (Vr_r / r) * (t2a + t2b + t2c)
    term3 = -2.0 * (Vr / r**2) * np.einsum('j,ki->kij', n, (I - np.outer(n, n)))
    return term1 + term2 + term3

# ---------- Extract Vr, Vr_r, Vr_rr for beta and lambda from your data ----------
beta_r   = Shift[0]
beta_r_r = d1Shift[0][0]             # radial derivative dβ_r/dr
beta_r_rr= d2Shift[0][0][0]         # second radial derivative d^2β_r/dr^2

# For Lambda you didn’t provide d^2/dr^2 of the radial comp; set 0 or your value if you have it:
lambda_r   = Lambda[0]
lambda_r_r = d1Lambda[0][0]          # careful: your dLambda also has extra rows; the first row,first col is V_r'

# ---------- Build full Cartesian derivatives ----------
d1Shift_cartesian  = grad_vector_full(beta_r,   beta_r_r,   r[10], n)   # 3x3 (∂_i β^j)
d2Shift_cartesian  = hess_vector_full(beta_r,   beta_r_r,   beta_r_rr,   r[10], n)  # 3x3x3 (∂_k∂_i β^j)
d1Lambda_cartesian = grad_vector_full(lambda_r, lambda_r_r, r[10], n)   # 3x3 (∂_i Λ^j)

# ---------- Show results ----------
np.set_printoptions(precision=12, suppress=True)

print("n =", n, " r =", r[10])

print("\nFULL Cartesian gradient ∂_i β^j:\n", d1Shift_cartesian)

print("\nFULL Cartesian gradient ∂_i Λ^j:\n", d1Lambda_cartesian)

print("\nFULL Cartesian second partials ∂_k∂_i β^j:\n", d2Shift_cartesian)



cartesian_names.append("dshift")
cartesian_objects.append(d1Shift_cartesian)

cartesian_names.append("dgamma")
cartesian_objects.append(d1Lambda_cartesian)

cartesian_names.append("d2shift")
cartesian_objects.append(d2Shift_cartesian)



"""
print("Slices (k=x,y,z):")
print("k=x:\n", H_beta[0])
print("k=y:\n", H_beta[1])
print("k=z:\n", H_beta[2])
"""

n = [1. 0. 0.]  r = 0.47993604100070275

FULL Cartesian gradient ∂_i β^j:
 [[-0.092758170872  0.              0.            ]
 [ 0.              0.190493423148  0.            ]
 [ 0.              0.              0.190493423148]]

FULL Cartesian gradient ∂_i Λ^j:
 [[-0.17841631739   0.              0.            ]
 [ 0.              1.003151003427  0.            ]
 [ 0.              0.              1.003151003427]]

FULL Cartesian second partials ∂_k∂_i β^j:
 [[[-0.464759423128 -0.             -0.            ]
  [-0.             -0.1932719424   -0.            ]
  [-0.             -0.             -0.1932719424  ]]

 [[-0.             -0.1932719424   -0.            ]
  [-0.987100314823 -0.             -0.            ]
  [-0.             -0.             -0.            ]]

 [[-0.             -0.             -0.1932719424  ]
  [-0.             -0.             -0.            ]
  [-0.987100314823 -0.             -0.            ]]]


'\nprint("Slices (k=x,y,z):")\nprint("k=x:\n", H_beta[0])\nprint("k=y:\n", H_beta[1])\nprint("k=z:\n", H_beta[2])\n'

# Tensors coordinate transformations

In [21]:
Gamma = initial[200][1][0][10]
A = initial[200][3][0][10]


def sym_tensor_from_sph_diag(Trr, Ttt, n, I, P):
    return Trr * np.outer(n, n) + Ttt * P

Gamma_cartesian = sym_tensor_from_sph_diag(Gamma[0][0], Gamma[1][1], n, I, P)
cartesian_names.append("h")
cartesian_objects.append(Gamma_cartesian)

A_cartesian = sym_tensor_from_sph_diag(A[0][0], A[1][1], n, I, P) 
cartesian_names.append("A")
cartesian_objects.append(A_cartesian)


# Derivatives of Tensors coordinate transformation

In [22]:
d1A = initial[200][15][0][10]
d1Gamma = initial[200][14][0][10]

d2Gamma = initial[200][21][0][10]

# First derivative of A
ARR = A[0][0]
ATT = A[1][1]
ARR_dr = d1A[0][0][0]
ATT_dr = d1A[0][1][1]

# First derivative of G
GRR = Gamma[0][0]
GTT = Gamma[1][1]
GRR_dr = d1Gamma[0][0][0]
GTT_dr = d1Gamma[0][1][1]

GRR_dr_dr = d2Gamma[0][0][0][0]
GTT_dr_dr = d2Gamma[0][0][1][1]

# Second derivative of G

def grad_tensor_from_sph_diag(Trr, Ttt, dTrr_dr, dTtt_dr, r, n, I, P):
    nn = np.outer(n, n)
    # term1: T_rr' n_k n_i n_j  -> store as a function returning ∂k T_ij
    # We'll build ∂k T_ij as a rank-3 array G[k,i,j]
    G = np.zeros((3,3,3))
    # Helper tensors
    # A_ik = δ_ik - n_i n_k ;  B_jk = δ_jk - n_j n_k
    # We'll construct per k.
    for k in range(3):
        e_k = np.zeros(3); e_k[k] = 1.0
        A = np.eye(3) - np.outer(n, e_k) @ np.outer(n, e_k).T  # (δ_ik - n_i n_k) as a 3x3 in i,k? We'll do it termwise below.
        # But easier: use arrays explicitly
        for i in range(3):
            for j in range(3):
                term1 = dTrr_dr * n[k] * n[i] * n[j]
                term2 = dTtt_dr * n[k] * (P[i, j])
                term3 = ((Trr - Ttt) / r) * (( (int(i==k) - n[i]*n[k]) * n[j] ) + ( (int(j==k) - n[j]*n[k]) * n[i] ))
                G[k, i, j] = term1 + term2 + term3
    return G  # shape (3,3,3) with indices [k,i,j] = ∂k 

d1A_cartesian = grad_tensor_from_sph_diag(ARR,ATT, ARR_dr, ATT_dr, r[10], n,I, P )
cartesian_names.append("dA")
cartesian_objects.append(d1A_cartesian)

d1Gamma_cartesian = grad_tensor_from_sph_diag(GRR,GTT, GRR_dr, GTT_dr, r[10], n,I, P )
cartesian_names.append("dh")
cartesian_objects.append(d1Gamma_cartesian)



def hess_tensor_from_sph_diag(Trr, Ttt, dTrr_dr, dTtt_dr, d2Trr_dr2, d2Ttt_dr2, r, n):
    """
    Second derivative: H[l,k,i,j] = ∂_l ∂_k T_ij.
    Assumes spherical symmetry with tangential degeneracy Ttt = Tpp.
    Inputs are the radial profiles and their r-derivatives at the evaluation radius r.
    """
    n = np.asarray(n, float)
    I = np.eye(3)
    # Use T = Q δ + S n n  with Q=Ttt, S=Trr-Ttt
    Qp, Qpp = dTtt_dr,  d2Ttt_dr2
    Sp, Spp = (dTrr_dr - dTtt_dr), (d2Trr_dr2 - d2Ttt_dr2)

    # helpers
    nn = np.outer(n, n)              # n_i n_j
    D  = I - nn                      # projector δ - n n

    # ∂_k n_i
    dn = np.zeros((3,3))             # dn[k,i] = ∂_k n_i
    for k in range(3):
        dn[k] = (I[:,k] - n * n[k]) / r

    # ∂_l∂_k n_i
    d2n = np.zeros((3,3,3))          # d2n[l,k,i] = ∂_l∂_k n_i
    for l in range(3):
        for k in range(3):
            # ∂_l∂_k n_i = - (n_l/r^2)(δ_{ik} - n_i n_k) + (1/r)[ - (∂_l n_i) n_k - n_i (∂_l n_k) ]
            d2n[l,k] = - (n[l]/r**2) * (I[:,k] - n * n[k]) + ( - dn[l] * n[k] - n * dn[l,k] ) / r

    H = np.zeros((3,3,3,3))          # [l,k,i,j]

    # Build terms:
    # A) (Q'' n_l n_k + Q' ∂_l n_k) δ_ij
    for l in range(3):
        for k in range(3):
            coefA = Qpp * n[l] * n[k] + Qp * dn[l, k]
            H[l, k] += coefA * I

    # B) (S'' n_l n_k + S' ∂_l n_k) n_i n_j
    for l in range(3):
        for k in range(3):
            coefB = Spp * n[l] * n[k] + Sp * dn[l, k]
            H[l, k] += coefB * np.einsum('i,j->ij', n, n)

    # C) S' n_k [ (∂_l n_i) n_j + n_i (∂_l n_j) ]
    for l in range(3):
        for k in range(3):
            H[l, k] += Sp * n[k] * (np.einsum('i,j->ij', dn[l], n) + np.einsum('i,j->ij', n, dn[l]))

    # D) S [ (∂_l∂_k n_i) n_j + (∂_k n_i)(∂_l n_j) + (∂_l n_i)(∂_k n_j) + n_i (∂_l∂_k n_j) ]
    for l in range(3):
        for k in range(3):
            termD = (np.einsum('i,j->ij', d2n[l,k], n) +
                     np.einsum('i,j->ij', dn[k],     dn[l]) +
                     np.einsum('i,j->ij', dn[l],     dn[k]) +
                     np.einsum('i,j->ij', n,         d2n[l,k]))
            H[l, k] += (Trr - Ttt) * termD

    return H  # shape (3,3,3,3) with indices [l,k,i,j] = ∂_l ∂_k T_ij

d2Gamma_cartesian = hess_tensor_from_sph_diag(GRR, GTT, GRR_dr, GTT_dr, GRR_dr_dr, GTT_dr_dr, r[10], n)
cartesian_names.append("d2h")
cartesian_objects.append(d2Gamma_cartesian)

In [23]:
for i in range(len(cartesian_names)):
    print(cartesian_names[i],"=", cartesian_objects[i] ,";")

chi = 0.0609867684537276 ;
K = 0.01693349180876882 ;
lapse = 0.9941885069169928 ;
phi = 6.750501947627349e-06 ;
Pi = 0.00013524018932775466 ;
dchi = [0.24956161159071186, -0.0, -0.0] ;
d2chi = [[0.24365894872302293, 0.0, 0.0], [1.0212214806398159, 0.0, 0.0], [1.0212214806398159, 0.0, 0.0]] ;
dphi = [0.0006706916626822981, 0.0, 0.0] ;
d2phi = [[0.03386028851327162, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]] ;
dPi = [0.008924708167139975, 0.0, 0.0] ;
dlapse = [-0.0005295522048608529, 0.0, 0.0] ;
d2lapse = [[0.067395183021676, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]] ;
dK = [-0.00014152793581781282, 0.0, 0.0] ;
gamma = [0.4814483211105061, 0.0, 0.0] ;
shift = [0.09142465934210317, 0.0, 0.0] ;
dshift = [[-0.092758170872  0.              0.            ]
 [ 0.              0.190493423148  0.            ]
 [ 0.              0.              0.190493423148]] ;
dgamma = [[-0.17841631739   0.              0.            ]
 [ 0.              1.003151003427  0.            ]
 [ 0.              

In [24]:
cartesian_names

['chi',
 'K',
 'lapse',
 'phi',
 'Pi',
 'dchi',
 'd2chi',
 'dphi',
 'd2phi',
 'dPi',
 'dlapse',
 'd2lapse',
 'dK',
 'gamma',
 'shift',
 'dshift',
 'dgamma',
 'd2shift',
 'h',
 'A',
 'dA',
 'dh',
 'd2h']

# conversion to mathematica code

In [25]:
import numpy as np

def to_mathematica_number(x: float) -> str:
    s = f"{x:.17g}"
    if s in ("-0", "-0.0"):  # normalize -0
        s = "0"
    if "e" in s or "E" in s:  # 1.23e-4 -> 1.23*^-4
        base, exp = s.replace("E", "e").split("e")
        if exp.startswith("+"):
            exp = exp[1:]
        s = f"{base}*^{exp}"
    return s

def to_mathematica(obj) -> str:
    if isinstance(obj, (np.floating, np.integer)):
        return to_mathematica_number(float(obj))
    if isinstance(obj, (int, float)):
        return to_mathematica_number(float(obj))
    if isinstance(obj, np.ndarray):
        return to_mathematica(obj.tolist())
    if isinstance(obj, (list, tuple)):
        return "{" + ", ".join(to_mathematica(x) for x in obj) + "}"
    return str(obj)

# ---- your data here ----
cartesian_objects
cartesian_names 

# print lines you can paste into Mathematica
for name, obj in zip(cartesian_names, cartesian_objects):
    print(f"{name} = {to_mathematica(obj)} ;")

# optional: write to a file (e.g., to avoid truncation in consoles)
with open("cartesian_assignments.m", "w") as f:
    for name, obj in zip(cartesian_names, cartesian_objects):
        f.write(f"{name} = {to_mathematica(obj)} ;\n")


chi = 0.060986768453727601 ;
K = 0.016933491808768821 ;
lapse = 0.99418850691699279 ;
phi = 6.7505019476273494*^-06 ;
Pi = 0.00013524018932775466 ;
dchi = {0.24956161159071186, 0, 0} ;
d2chi = {{0.24365894872302293, 0, 0}, {1.0212214806398159, 0, 0}, {1.0212214806398159, 0, 0}} ;
dphi = {0.00067069166268229809, 0, 0} ;
d2phi = {{0.033860288513271622, 0, 0}, {0, 0, 0}, {0, 0, 0}} ;
dPi = {0.0089247081671399754, 0, 0} ;
dlapse = {-0.00052955220486085287, 0, 0} ;
d2lapse = {{0.067395183021676006, 0, 0}, {0, 0, 0}, {0, 0, 0}} ;
dK = {-0.00014152793581781282, 0, 0} ;
gamma = {0.48144832111050612, 0, 0} ;
shift = {0.091424659342103171, 0, 0} ;
dshift = {{-0.092758170871821935, 0, 0}, {0, 0.1904934231475425, 0}, {0, 0, 0.1904934231475425}} ;
dgamma = {{-0.17841631739018654, 0, 0}, {0, 1.0031510034267277, 0}, {0, 0, 1.0031510034267277}} ;
d2shift = {{{-0.46475942312765745, 0, 0}, {0, -0.19327194239968762, 0}, {0, 0, -0.19327194239968762}}, {{0, -0.19327194239968762, 0}, {-0.98710031482343541, 

# RHS Values

In [26]:

idx = 10  #Position index
idx_t = 200  #time index

print( "time: ", t[idx_t],"position: ", r[idx],)
print(' ')
#idx = NUM_GHOSTS + 2 # Choose an inner point
#r_i = np.round(r[idx],2)

for i in range(len(my_state_vector.VARIABLE_NAMES)):
    var1_of_t = solution[:, i * num_points + idx]
    print("Final value for   ",my_state_vector.VARIABLE_NAMES[i],"   :   ",var1_of_t[idx_t])



time:  0.8 position:  0.47993604100070275
 
Final value for    phi    :    0.7002502904821324
Final value for    hrr    :    0.05922546668629869
Final value for    htt    :    -0.028358953041447227
Final value for    hpp    :    -0.028358953041447227
Final value for    K    :    0.01570107910062095
Final value for    arr    :    -0.21553803584774928
Final value for    att    :    0.09885540468260658
Final value for    app    :    0.09885540468260658
Final value for    lambdar    :    0.47267093890771167
Final value for    shiftr    :    0.08710870105939966
Final value for    br    :    0.24643517421372763
Final value for    lapse    :    0.9947459864800955
Final value for    u    :    4.949719818857592e-06
Final value for    v    :    9.046397661878372e-05


In [27]:
chik = np.exp(-4*solution[:, 0 * num_points + 10][200])
chik

0.060749212371207575

In [28]:
def rhs_at(final, idx_t, idx):
    snap = final[idx_t]
    return {
        "chi":   np.exp(-4*snap["phi_rhs"][idx]),
        "K":     snap["K_rhs"][idx],
        "lapse": snap["lapse_rhs"][idx],
        "lambda": snap["lambda_U_rhs"][idx],   # len 3
        "shift":  snap["shift_U_rhs"][idx],    # len 3
        "h":      snap["h_LL_rhs"][idx],       # 3x3
        "A":      snap["a_LL_rhs"][idx],       # 3x3
        "u":      snap["u"][idx],
        "Pi":      snap["Pi"][idx],
        
    }

res = rhs_at(final, idx_t, idx)

# 1) One key per line (simple)
for k, v in res.items():
    print(f"{k}: {v}")


chi: 1.211795031011433
K: 0.06121995836697933
lapse: -0.028058471563751672
lambda: [0.4180685625238019, 0.0, 0.0]
shift: [0.20948558823556795, 0.0, 0.0]
h: [[0.052573648663849865, -0.04988880383716218, -0.04988880383716218], [-0.04988880383716218, -0.024076928106442384, 0.0], [-0.04988880383716218, 0.0, -0.024076928106442384]]
A: [[-0.29318559431565755, 0.08794898510289044, 0.08794898510289044], [0.08794898510289044, 0.02918403140882319, 0.0], [0.08794898510289044, 0.0, 0.02918403140882319]]
u: 0.00011896530685792165
Pi: 0.0024681776013169227


In [ ]:
double dshiftdt_known[3] = {0.20948243777170822, 0., 0.};
                    shift: [0.20948558823556795, 0.0, 0.0]
